In [1]:
from neo4j import GraphDatabase
import dotenv
import os

dotenv.load_dotenv()

URI = "neo4j://localhost:7687"
AUTH = ("neo4j", os.getenv("NEO4J_PASSWORD"))

In [2]:
# with GraphDatabase.driver(URI, auth=AUTH) as driver:
#     print("Connected to Neo4j")
#     summary = driver.execute_query(
#         """
#         CREATE (a:Person {name: $name})
#         CREATE (b:Person {name: $friendName})
#         CREATE (a)-[:KNOWS]->(b)
#         """,
#         name="Alice", friendName="David",
#     ).summary
#     print("Created {nodes_created} nodes in {time} ms.".format(
#         nodes_created=summary.counters.nodes_created,
#         time=summary.result_available_after
#     ))

In [3]:
with GraphDatabase.driver(URI, auth=AUTH) as driver:
    print("Connected to Neo4j")
    records, summary, keys = driver.execute_query(
        """
        MATCH (p:Person{name: $name})-[r:DIRECTED]->(m:Movie)
        WHERE m.year > $year
        RETURN p.name AS director, m.year AS year, m.title AS movie
        """,
        # name="张艺谋", year=1990,
        parameters_={"name": "张艺谋", "year": 1990},
    )
    print(f"查询的结果为：{len(records)}, 运行时间：{summary.result_available_after}ms")
    for record in records:
        print(record.data())

Connected to Neo4j
查询的结果为：3, 运行时间：1ms
{'director': '张艺谋', 'year': 2023, 'movie': '满江红'}
{'director': '张艺谋', 'year': 2002, 'movie': '英雄'}
{'director': '张艺谋', 'year': 1994, 'movie': '活着'}


In [4]:
for record in records:
    print(f"{record['director']}在 {record['year']}年 导演了 {record['movie']}")

张艺谋在 2023年 导演了 满江红
张艺谋在 2002年 导演了 英雄
张艺谋在 1994年 导演了 活着
